In [17]:
import io
import sqlite3
from contextlib import closing
from datetime import date, timedelta
from html import escape
from pathlib import Path
import pandas as pd
import streamlit as st
import numpy as np

In [29]:
# ====== KONFIGURASI ======
BASE_DIR = Path.cwd()  # folder kerja notebook (pengganti __file__ di Jupyter)
DB_PATH = (
    BASE_DIR
    / ".."
    / "cleaning_data"
    / "3 Oktober 2026"
    / "Mikro"
    / "RantabDBNAS"
    / "cleaning.db"
).resolve()


TABEL = "usaha"
KOL_KATEGORI = "kategori"
KOL_RASIO = "rasio_ntb"
DESIMAL = 2
TOLERANSI = 0.01  # ambang +- 0.01

# kategori: (batas_bawah, batas_atas)
BATAS = {
'A': (0.51, 0.94),
'B': (0.28, 0.81),
'C': (0.15, 0.61),
'D': (0.13, 0.68),
'E': (0.29, 0.47),
'F': (0.34, 0.44),
'G': (0.64, 0.78),
'H': (0.29, 0.64),
'I': (0.36, 0.69),
'J': (0.36, 0.73),
'K': (0.5, 0.72),
'L': (0.46, 0.86),
'M': (0.5, 0.99),
'N': (0.44, 0.57),
'O': (0.54, 0.66),
'Q': (0.27, 0.82),
'R': (0.2, 0.6),
'S': (0.21, 0.57),
'T': (0.3, 0.69),

}

FILTER_UTAMA = """
    (CAST(flag_rasio_3_ntb_output AS INTEGER) = 1
     OR CAST(metrik_nilai_tambah AS REAL) < 0)
"""

In [15]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

with sqlite3.connect(DB_PATH) as conn:
    # 1. Semua tabel di database
    tabel = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY name",
        conn,
    )
    print("TABEL:")
    print(tabel.to_string(index=False))

    # 2. Nama kolom + tipe data tabel usaha
    kolom = pd.read_sql_query("PRAGMA table_info(usaha)", conn)
    print("\nKOLOM tabel usaha:")
    print(kolom[["cid", "name", "type"]].to_string(index=False))

    # 3. Jumlah baris
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM usaha", conn)["n"][0]
    print(f"\nJumlah baris: {n}")

    # 4. Contoh 5 baris, semua kolom
    contoh = pd.read_sql_query("SELECT * FROM usaha LIMIT 5", conn)

contoh

TABEL:
 name
usaha

KOLOM tabel usaha:
 cid                                        name    type
   0                                        link    TEXT
   1                               assignment_id    TEXT
   2                                level_2_code    TEXT
   3                                    kategori    TEXT
   4                                   kode_kbli    TEXT
   5                                  nama_usaha    TEXT
   6                         metrik_tenaga_kerja INTEGER
   7                            metrik_upah_gaji INTEGER
   8                       metrik_biaya_produksi INTEGER
   9                      metrik_biaya_pembelian INTEGER
  10                    metrik_biaya_operasional INTEGER
  11                metrik_biaya_non_operasional INTEGER
  12                    metrik_total_pengeluaran INTEGER
  13               metrik_pendapatan_barang_jasa INTEGER
  14                   metrik_pendapatan_lainnya INTEGER
  15                         metrik_nilai_tambah 

,link,assignment_id,level_2_code,kategori,kode_kbli,nama_usaha,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,metrik_biaya_non_operasional,metrik_total_pengeluaran,metrik_pendapatan_barang_jasa,metrik_pendapatan_lainnya,metrik_nilai_tambah,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,reklasifikasi_skala_usaha,rasio_ntb,produktivitas,flag_rasio_1_output_aset,flag_rasio_2_upah_ntb,flag_rasio_3_ntb_output,flag_kategori_P_dan_U,sumber_file,status_penyelesaian,keterangan,update_at
0,https://fasih-sm.bps.go.id/app/assignment/fd68...,0067eab8-97e4-4169-940c-6b7d124b102f,1601,A,01111,PERKEBUNAN JAGUNG MARDANI,1,0,35000000,0,8000000,0,43000000,86000000,0,43000000,5000000,86000000,0.0,Mikro,0.500000,8.600000e+07,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,2026-10-06 11:07:20
1,https://fasih-sm.bps.go.id/app/assignment/fd68...,00754e8b-9421-4536-ac01-778f0ee3b0c4,1601,A,01111,UTP TANAMAN PANGAN (ISMANTO),2,0,14880000,0,8160000,0,23040000,0,0,-23040000,0,0,NaN,Mikro,NaN,0.000000e+00,None,1,None,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,1,Sesuai kondisi,2026-10-06 20:12:37
2,https://fasih-sm.bps.go.id/app/assignment/fd68...,008baf68-cf1b-44a5-9015-3ef04664306e,1601,A,01111,UTP TANAMAN PANGAN JAGUNG (MARKUS WARJU),7,5000000,7080000,0,1500000,0,13580000,59000000,0,50420000,115000000,59000000,0.0,Mikro,0.854576,8.428571e+06,1,1,0,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None
3,https://fasih-sm.bps.go.id/app/assignment/fd68...,00a48eda-54fa-427f-8fcb-a1ebf42440bf,1601,A,01111,PERKEBUNAN JAGUNG (ASMARA),1,0,4000000,0,3000000,500000,7500000,12100000,0,5100000,0,12100000,0.0,Mikro,0.421488,1.210000e+07,None,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None
4,https://fasih-sm.bps.go.id/app/assignment/fd68...,00e1b203-ea5a-4c97-832c-221264a478a2,1601,A,01111,PERKEBUNAN JAGUNG,2,0,24000000,0,6000000,2000000,32000000,45000000,0,15000000,1000000,45000000,0.0,Mikro,0.333333,2.250000e+07,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None


In [30]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"File database tidak ditemukan: {DB_PATH}")

query = f"SELECT * FROM {TABEL} WHERE {FILTER_UTAMA}"

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(query, conn)

print(f"Lolos filter utama (flag = 1 OR nilai tambah < 0): {len(df)} baris")

# --- Filter ambang rasio per kategori ---
kat = df[KOL_KATEGORI].astype(str).str.strip().str.upper()
rasio = pd.to_numeric(df[KOL_RASIO], errors="coerce")

bawah = kat.map(lambda k: BATAS.get(k, (np.nan, np.nan))[0])
atas = kat.map(lambda k: BATAS.get(k, (np.nan, np.nan))[1])

rasio_bulat = rasio.round(DESIMAL)

# Nilai setelah dibulatkan = batas, tapi nilai aslinya BUKAN persis batas
# (artinya hanya "kelihatan" sama karena pembulatan)
persis_bawah = np.isclose(rasio, bawah, rtol=0, atol=1e-9)
persis_atas = np.isclose(rasio, atas, rtol=0, atol=1e-9)

kena_bawah = np.isclose(rasio_bulat, bawah, rtol=0, atol=1e-9) & ~persis_bawah
kena_atas = np.isclose(rasio_bulat, atas, rtol=0, atol=1e-9) & ~persis_atas

hasil = df[kena_bawah | kena_atas].copy()
idx = hasil.index

Lolos filter utama (flag = 1 OR nilai tambah < 0): 840291 baris


In [31]:
hasil["rasio_asli"] = rasio[idx]
hasil["rasio_dibulatkan"] = rasio_bulat[idx]
hasil["batas_bawah"] = bawah[idx]
hasil["batas_atas"] = atas[idx]
hasil["batas_terkena"] = np.where(kena_bawah[idx], "bawah", "atas")
hasil["batas_nilai"] = np.where(kena_bawah[idx], bawah[idx], atas[idx])
hasil["selisih_dari_batas"] = hasil["rasio_asli"] - hasil["batas_nilai"]
# Nilai asli ada DI DALAM rentang (bawah < rasio < atas) tapi tetap kena batas karena pembulatan
hasil["asli_di_dalam_rentang"] = (hasil["rasio_asli"] > hasil["batas_bawah"]) & (
    hasil["rasio_asli"] < hasil["batas_atas"]
)

hasil = hasil.sort_values([KOL_KATEGORI, "batas_terkena", "selisih_dari_batas"])

print(f"Hasil akhir (rasio dibulatkan sama dengan batas): {len(hasil)} baris")
print(hasil.groupby([KOL_KATEGORI, "batas_terkena"]).size().to_string())
print("\nAsli di dalam rentang:", hasil["asli_di_dalam_rentang"].sum(), "baris")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

hasil.to_excel(BASE_DIR / "usaha_rasio_dibulatkan.xlsx", index=False)
hasil  # tampilkan di Jupyter

Hasil akhir (rasio dibulatkan sama dengan batas): 35307 baris
kategori  batas_terkena
A         atas             26156
          bawah             1982
B         atas                23
          bawah                9
C         atas               632
          bawah              166
E         atas                11
          bawah                7
F         atas                18
G         atas              3274
          bawah              853
H         atas                 3
          bawah              212
I         atas               223
          bawah             1041
J         atas                 2
          bawah                1
K         atas                46
          bawah               10
L         atas                26
          bawah                8
M         atas                71
          bawah                6
N         atas                10
          bawah                9
O         atas                36
          bawah               23
Q         atas         

,link,assignment_id,level_2_code,kategori,kode_kbli,nama_usaha,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,metrik_biaya_non_operasional,metrik_total_pengeluaran,metrik_pendapatan_barang_jasa,metrik_pendapatan_lainnya,metrik_nilai_tambah,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,reklasifikasi_skala_usaha,rasio_ntb,produktivitas,flag_rasio_1_output_aset,flag_rasio_2_upah_ntb,flag_rasio_3_ntb_output,flag_kategori_P_dan_U,sumber_file,status_penyelesaian,keterangan,update_at,rasio_asli,rasio_dibulatkan,batas_bawah,batas_atas,batas_terkena,batas_nilai,selisih_dari_batas,asli_di_dalam_rentang
372153,https://fasih-sm.bps.go.id/app/assignment/fd68...,c4143a35-a468-4569-a9a0-54f37054e971,1607,A,01261,USAHA PERKEBUNAN KELAPA (SUPARDI),2,1250000,457000,0,123000,0,1830000,9547000,250000,8967000,55000000,9547000,0.000000,Mikro,0.939248,4773500.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.939248,0.940000,0.510000,0.940000,atas,0.940000,-0.000752,True
146870,https://fasih-sm.bps.go.id/app/assignment/fd68...,1e708f70-176a-4232-b79a-1910fa702e68,1603,A,01291,UTP PERKEBUNAN KARET (SANGKUT),2,0,660000,0,2256000,0,2916000,48000000,0,45084000,30500000,48000000,0.000000,Mikro,0.939250,24000000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.939250,0.940000,0.510000,0.940000,atas,0.940000,-0.000750,True
155802,https://fasih-sm.bps.go.id/app/assignment/fd68...,886ffe44-d65b-4b63-9dc7-ed0cf3442bdc,1603,A,01291,UTP PERKEBUNAN KARET (MANAN ARIANTO),2,0,660000,0,2256000,20000000,22916000,48000000,0,45084000,475000,48000000,0.000000,Mikro,0.939250,24000000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.939250,0.940000,0.510000,0.940000,atas,0.940000,-0.000750,True
163455,https://fasih-sm.bps.go.id/app/assignment/fd68...,e336f92e-d899-44d6-a6f9-8cc8263d0589,1603,A,01291,UTP PERKEBUNAN KARET (DANI),2,0,660000,0,2256000,0,2916000,48000000,0,45084000,35530000,48000000,0.000000,Mikro,0.939250,24000000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.939250,0.940000,0.510000,0.940000,atas,0.940000,-0.000750,True
213996,https://fasih-sm.bps.go.id/app/assignment/fd68...,59cde822-b3f1-44b4-8119-7a94e6704390,1604,A,01271,KEBUN KOPI (EFRI),1,0,1230000,0,1200000,0,2430000,40000000,0,37570000,151000000,40000000,0.000000,Mikro,0.939250,40000000.000000,0,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.939250,0.940000,0.510000,0.940000,atas,0.940000,-0.000750,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800840,https://fasih-sm.bps.go.id/app/assignment/fd68...,5245d36c-c4d2-45d2-8ff1-f8719952ccca,1672,T,95311,BENGKEL ERIK,1,0,58500000,0,4500000,8500000,71500000,89375000,4000000,26375000,180000000,89375000,0.000000,Mikro,0.295105,89375000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.295105,0.300000,0.300000,0.690000,bawah,0.300000,-0.004895,False
778433,https://fasih-sm.bps.go.id/app/assignment/fd68...,42df4926-f0b1-4890-b7df-bc6f2d932e18,1671,T,96100,KITAA LAUNDRY,2,18000000,11400000,0,33500000,800000,63700000,63700000,0,18800000,20000000,63700000,0.000000,Mikro,0.295133,31850000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.295133,0.300000,0.300000,0.690000,bawah,0.300000,-0.004867,False
640925,https://fasih-sm.bps.go.id/app/assignment/fd68...,d1b547e0-a097-43b5-998e-6c17c6ffc8fa,1612,T,95311,BENGKEL MOBIL,3,4800000,96000000,0,5500000,0,106300000,144000000,0,42500000,85000000,144000000,0.000000,Mikro,0.295139,48000000.000000,1,1,1,0,Tabel 3 (MIKRO). Usaha Tenaga Kerja Pengeluara...,0,,None,0.295139,0.300000,0.300000,0.690000,bawah,0.300000,-0.004861,False
778369,https://fasih-sm.bps.go.id/app/assignment/fd68...,2c9198b7-d5b5-4c01-a719-b1afb96fb2dd,1671,T,96100,PENJAHIT LOANDRY A